In [ ]:
import torch
torch.cuda.is_available()


True

In [ ]:
!pip install -q transformers datasets torch torchvision pillow accelerate


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_CHECKPOINT = "microsoft/trocr-base-handwritten"

processor = TrOCRProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_CHECKPOINT)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import os
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/iam words/iam_words"
WORDS_TXT = os.path.join(BASE_PATH, "words.txt")
WORDS_DIR = os.path.join(BASE_PATH, "words")

data = []

with open(WORDS_TXT, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue

        parts = line.strip().split()

        # Ensure there are enough parts to avoid IndexError
        # The code expects at least 9 parts (up to index 8 for text)
        if len(parts) < 9:
            continue

        word_id = parts[0]
        status = parts[1]

        if status != "ok":
            continue

        text = " ".join(parts[8:])

        folder1 = word_id.split("-")[0]
        folder2 = "-".join(word_id.split("-")[:2])

        image_path = f"{WORDS_DIR}/{folder1}/{folder2}/{word_id}.png"

        if os.path.exists(image_path):
            data.append({
                "image_path": image_path,
                "text": text
            })

df = pd.DataFrame(data)
print("Total IAM samples:", len(df))
df.head()
# 🔥 LIMIT DATASET SIZE (VERY IMPORTANT)
df = df.sample(n=2500, random_state=42)
print("Using samples:", len(df))


Total IAM samples: 7574
Using samples: 2500


In [ ]:
from datasets import Dataset
from PIL import Image

dataset = Dataset.from_pandas(df)
print(dataset)


Dataset({
    features: ['image_path', 'text', '__index_level_0__'],
    num_rows: 2500
})


In [ ]:
from PIL import Image

def is_valid_image(example):
    try:
        img = Image.open(example["image_path"])
        img.verify()   # checks corruption
        return True
    except:
        return False

dataset = dataset.filter(is_valid_image)
print("Valid images:", len(dataset))


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Valid images: 2500


In [ ]:
from PIL import Image

def preprocess(example):
    try:
        image = Image.open(example["image_path"])

        if image.mode != "RGB":
            image = image.convert("RGB")

        image = image.resize((384, 384))

        pixel_values = processor(
            images=image,
            return_tensors="pt"
        ).pixel_values[0]

        labels = processor.tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=64
        ).input_ids

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

    except Exception as e:
        return {
            "pixel_values": None,
            "labels": None
        }


In [ ]:
processed_dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./trocr_iam_words",

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch = 16

    num_train_epochs=5,
    learning_rate=2e-5,

    fp16=True,                       # ✅ ONLY if GPU is enabled
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,

    dataloader_num_workers=2,        # 🚀 GPU pipeline speedup
    report_to="none"
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    tokenizer=processor
)
trainer.train()


In [ ]:
model.save_pretrained("handwritten_trocr_model")
processor.save_pretrained("handwritten_trocr_model")


In [ ]:
!zip -r handwritten_trocr_model.zip handwritten_trocr_model


In [ ]:
from google.colab import files
files.download("handwritten_trocr_model.zip")
